# Skillnox AI — Fine-Tune Qwen3-8B on Kaggle
**Model**: Qwen3-8B (text-only, latest gen)
**Method**: QLoRA (4-bit) via Unsloth
**GPU**: Kaggle T4 (single GPU)
**Training time**: ~2-3 hours

## Setup
1. Accelerator: **GPU T4 x1** (Unsloth uses single GPU only)
2. Internet: **ON**
3. Add your `skillnox-training-data` dataset as input

## Cell 1 — Install Dependencies

In [ ]:
!pip install -U unsloth transformers datasets peft accelerate bitsandbytes trl safetensors huggingface_hub
print("\n=== Dependencies installed! ===")

## Cell 2 — Download Training Data (if not added as input)

In [ ]:
import os

# Check if dataset is available as Kaggle input
DATA_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".jsonl"):
            DATA_PATH = os.path.join(root, f)
            print(f"Found dataset: {DATA_PATH}")
            break

# If not found, download it
if DATA_PATH is None:
    print("Dataset not found in /kaggle/input, downloading...")
    !kaggle datasets download surendravarikallu1/skillnox-training-data -p /kaggle/working/data --unzip
    for f in os.listdir("/kaggle/working/data"):
        if f.endswith(".jsonl"):
            DATA_PATH = f"/kaggle/working/data/{f}"
            break

print(f"\nUsing dataset: {DATA_PATH}")

## Cell 3 — Load Qwen3-8B with QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"  # Pre-quantized, text-only
MAX_SEQ_LEN = 1024

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # LoRA rank
    lora_alpha=32,        # LoRA scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 60% less VRAM
    random_state=42,
)

print("\n=== Model loaded with LoRA adapters ===")
model.print_trainable_parameters()

## Cell 4 — Prepare Training Dataset

In [ ]:
import json
import random
from datasets import Dataset

# Load raw training data
raw_data = []
with open(DATA_PATH) as f:
    for line in f:
        raw_data.append(json.loads(line))
print(f"Loaded {len(raw_data)} total examples")

# Skillnox system prompt
SYSTEM_PROMPT = (
    "You are SkillnoxAI, an expert AI-powered interview preparation and placement assistant. "
    "Your capabilities include: Interview Question Generation, Answer Evaluation (score 0-100), "
    "Resume Analysis, and Communication Assessment. "
    "Be STRICT but constructive. Do NOT inflate scores. "
    "For evaluations, always use: Score: [number]\nFeedback: [text]. "
    "Focus on Indian campus placement context (TCS, Infosys, Wipro, Accenture, etc.)."
)

# Format each example into ChatML
def format_example(example):
    instruction = example["instruction"]
    inp = json.dumps(example["input"]) if isinstance(example["input"], dict) else str(example["input"])
    out = json.dumps(example["output"]) if isinstance(example["output"], dict) else str(example["output"])
    
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
        f"<|im_start|>assistant\n{out}<|im_end|>"
    )
    return {"text": text}

# Shuffle and select 8000 examples (balanced training in ~2-3 hrs)
random.seed(42)
random.shuffle(raw_data)
train_data = raw_data[:8000]

# Count types
types = {}
for ex in train_data:
    t = ex["instruction"].split()[0]
    types[t] = types.get(t, 0) + 1
print(f"\nTraining distribution:")
for t, c in sorted(types.items(), key=lambda x: -x[1]):
    print(f"  {t}: {c}")

formatted = [format_example(ex) for ex in train_data]
dataset = Dataset.from_list(formatted)

print(f"\n=== Dataset ready: {len(dataset)} examples ===")
print(f"\nSample (first 300 chars):\n{formatted[0]['text'][:300]}...")

## Cell 5 — Configure and Start Training (~2-3 hours)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "/kaggle/working/output"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=True,
        logging_steps=25,
        save_steps=200,
        save_total_limit=2,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
)

print("=== Starting training ===")
print(f"Epochs: 3 | Batch: 2 | Grad Accum: 4 | Effective batch: 8")
print(f"LR: 2e-4 | Optimizer: AdamW 8-bit")
print(f"Checkpoints saved to: {OUTPUT_DIR}")
print("="*50)

stats = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"Total steps: {stats.global_step}")
print(f"Final loss: {stats.training_loss:.4f}")

# Save final LoRA adapter
model.save_pretrained("/kaggle/working/lora_adapter")
tokenizer.save_pretrained("/kaggle/working/lora_adapter")
print(f"LoRA adapter saved to /kaggle/working/lora_adapter")

## Cell 6 — Test the Fine-Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

def test_model(system, user_msg):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")
    out = model.generate(inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("TEST 1: Question Generation")
print("=" * 60)
q = test_model(
    "You are SkillnoxAI. Generate ONE technical interview question.",
    '{"question_type": "technical", "context": "Python backend development"}'
)
print(f"Output: {q}\n")

print("=" * 60)
print("TEST 2: Answer Evaluation")
print("=" * 60)
e = test_model(
    "You are SkillnoxAI. Evaluate with Score (0-100) and Feedback.",
    '{"question": "What is OOP?", "answer": "OOP stands for Object Oriented Programming. It has classes and objects."}'
)
print(f"Output: {e}\n")

print("=" * 60)
print("TEST 3: Resume Analysis")
print("=" * 60)
r = test_model(
    "You are SkillnoxAI. Analyze the resume.",
    '{"resume_text": "Amit Kumar\\nBackend Developer\\nSkills: Python, Django, PostgreSQL\\nExperience: 2 years\\nProjects: REST API gateway, Auth microservice"}'
)
print(f"Output: {r}\n")

print("=" * 60)
print("TEST 4: General Knowledge (sanity check)")
print("=" * 60)
g = test_model(
    "You are a helpful assistant.",
    "What is Python? Answer in 2 sentences."
)
print(f"Output: {g}")

## Cell 7 — Export to GGUF (for Ollama deployment)

In [ ]:
import os

GGUF_DIR = "/kaggle/working/skillnox-gguf"

print("Exporting to GGUF (q4_k_m quantization)...")
print("This takes ~10-15 minutes.\n")

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print("\n=== GGUF Export Complete ===")
# Find and report the GGUF file
for f in os.listdir(GGUF_DIR):
    fpath = os.path.join(GGUF_DIR, f)
    size_gb = os.path.getsize(fpath) / (1024**3)
    print(f"  {f}: {size_gb:.2f} GB")

print("\nDownload from the Output tab on the right sidebar.")
print("Then on your local machine:")
print("  1. Put the .gguf file in python-ai/models/")
print("  2. Update Modelfile line 1: FROM ./your-file.gguf")
print("  3. Run: ollama create skillnox-qwen:latest -f models/Modelfile")
print("  4. Start server: start_service.bat")

## Cell 8 — (Optional) Also save LoRA adapter to zip for backup

In [ ]:
import shutil

# Zip the LoRA adapter for backup
shutil.make_archive("/kaggle/working/skillnox-lora-adapter", "zip", "/kaggle/working/lora_adapter")
size_mb = os.path.getsize("/kaggle/working/skillnox-lora-adapter.zip") / (1024**2)
print(f"LoRA adapter backup: skillnox-lora-adapter.zip ({size_mb:.1f} MB)")
print("Download this too as backup — you can re-merge later without retraining.")